# Find Geographic Clusters

Start by importing the necessary libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Lasso, Ridge, LassoCV, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.linear_model import lasso_path
from sklearn.compose import make_column_transformer
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LinearRegression, Ridge, RidgeCV
from sklearn.linear_model import ElasticNet, ElasticNetCV
from sklearn.cluster import KMeans
import plotly.express as px
import plotly.graph_objects as go

Import data.

In [2]:
wildfires_all = pd.read_csv('../data/wildfires_clean2024.csv')
wildfires_active = wildfires_all[wildfires_all['Wildfire'] == 'Yes']

Perform K-Means on data to find clusters based on geographic coordinates.

In [3]:
wildfires_coords = wildfires_active[['latitude', 'longitude']]
kmeans = KMeans(n_clusters=6, random_state=30, n_init="auto").fit(wildfires_coords)

In [4]:
wildfires_clustered = wildfires_active.assign(cluster = kmeans.labels_)

Plot resulting clusters.

In [5]:
fig = px.scatter_map(wildfires_clustered, lat="latitude", lon="longitude", color="cluster",
                  color_continuous_scale=px.colors.cyclical.Phase, size_max=15, zoom=3)
fig.show()

In [6]:
wildfire_regions = wildfires_clustered.assign(
    region = wildfires_clustered['cluster'].map({
        0: 'Central West', 1: 'Northeast', 2: 'Southeast / Central', 3: 'Mountain-Plains', 4: 'Southwest', 5: 'Northwest'
    }))

fig = px.scatter_map(
    wildfire_regions, lat="latitude", lon="longitude", color="region",
    center={'lat': 39.8283, 'lon':-98.5795}, # Center on the US
    color_continuous_scale=px.colors.cyclical.Phase, size_max=15, zoom=3,
    title="Wildfire Labeled Geographic Clusters",
    subtitle="K-means clustering of wildfire ignition events based on geographic coordinates.",
    labels={"region": "Wildfire Region", 'latitude': "Latitude"})

# Add x/y axis labels 
fig.add_annotation(
    text="Longitude →", xref="paper", yref="paper",
    x=0.5, y=-0.05, showarrow=False, font=dict(size=12)
)
fig.add_annotation(
    text="↑ Latitude", xref="paper", yref="paper",
    x=-0.05, y=0.5, showarrow=False, textangle=-90, font=dict(size=12)
)

fig.show()

Save resulting clustered regions.

In [7]:
wildfire_regions.to_csv("../data/wildfires_regions2024.csv")